In [ ]:
import os, sys
from pathlib import Path
# Force JAX to ignore TPU/GPU backends in this environment.
os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.environ.setdefault("JAX_PLATFORM_NAME", "cpu")
import numpy as np
import scipy.optimize as sp_opt
import jax
jax.config.update("jax_enable_x64", True)
jax.config.update("jax_platform_name", "cpu")
jnp = jax.numpy
jsp = jax.scipy
random = jax.random
grad = jax.grad
jit = jax.jit
vmap = jax.vmap
import pickle
import tensorflow_probability.substrates.jax as tfp
tfpd = tfp.distributions
import optimistix as optx

In [2]:
@jit
def test_function(variables, args):
    x, y, z = variables
    a, b, c = args
    u = jnp.sin(a*x) + jnp.cos(b*y) + jnp.exp(c*z)
    v = jnp.tanh(x) + jnp.sinh(y) + jnp.sqrt(a**2 + b**2 + c**2)*jnp.cosh(z)
    w = x**2 + y**2 + z**2 - a*b*c
    return jnp.array([u, v, w])

MACHINE_EPSILON = sys.float_info.epsilon

In [3]:
a, b, c = 1.0, 2.0, 3.0
x_true, y_true, z_true = 10.0, 5.0, 0.0
u_meas, v_meas, w_meas = test_function((x_true, y_true, z_true), (a, b, c))
measured = jnp.array([u_meas, v_meas, w_meas])
print(f"Measured values: u={u_meas}, v={v_meas}, w={w_meas}")

@jit
def noneq_system(variables, args):
    return test_function(variables, args) - measured

initial_guess = (-10.0, 20.0, 1.0)
solver = optx.LevenbergMarquardt(rtol=MACHINE_EPSILON, atol=MACHINE_EPSILON)
solution = optx.root_find(noneq_system, solver, initial_guess, args=(a, b, c))
print(f"Solution: {solution.value}")
print(f"True values: {jnp.array([x_true, y_true, z_true])}")
print(f"System at solution: {noneq_system(solution.value, (a, b, c))}")
print(f"Function at solution: {test_function(solution.value, (a, b, c))}") 

Measured values: u=-0.3830926399658221, v=78.94486796044038, w=119.0
Solution: (Array(10., dtype=float64), Array(5., dtype=float64), Array(2.29470327e-17, dtype=float64))
True values: [10.  5.  0.]
System at solution: [0. 0. 0.]
Function at solution: [ -0.38309264  78.94486796 119.        ]


In [4]:
@jit
def skewnormal_cdf(x, loc=0.0, scale=1.0, shape=0.0, name=None):
    """Cumulative distribution function of the skew normal distribution."""
    inv_scale = 1.0 / scale
    t =(x - loc) * inv_scale
    return jsp.stats.norm.cdf(t, loc=0.0, scale=1.0) -2*tfp.math.owens_t(t,shape, name=name)

@jit
def logskewnormal_cdf(x, loc=0.0, scale=1.0, shape=0.0, name=None):
    return skewnormal_cdf(jnp.log(x), loc, scale, shape, name=name)

@jit
def logskewnormal_mode_condition(params, args):
    """Defines the function that has to be zero at the mode of the LogSkewNormal"""
    loc, scale, shape = params
    log_mode =  args
    inv_scale = 1.0 / scale
    t = (log_mode - loc) * inv_scale
    return shape * jnp.exp(-0.5 * (shape*t)**2) / jnp.sqrt(2 * jnp.pi) - t * jsp.stats.norm.cdf(shape * t)

@jit
def skewnormal_nonlin_system(params, args):
    loc, scale, shape = params
    log_mode, log_left_edge, log_right_edge = args
    mode_condition = logskewnormal_mode_condition(params, log_mode)
    left_edge_condition = logskewnormal_cdf(log_mode, loc=loc, scale=scale, shape=shape) - logskewnormal_cdf(
        log_left_edge, loc=loc, scale=scale, shape=shape) - 0.5 * jsp.special.erf(1/jnp.sqrt(2))
    right_edge_condition = logskewnormal_cdf(log_right_edge, loc=loc, scale=scale, shape=shape) - logskewnormal_cdf(
        log_mode, loc=loc, scale=scale, shape=shape) - 0.5 * jsp.special.erf(1/jnp.sqrt(2))
    return jnp.array([mode_condition, left_edge_condition, right_edge_condition])


In [12]:
from utils import import_bh_data
DEBUG = False

bh_data = import_bh_data("../data/bh_table_1.txt")
print(f"Loaded columns: {list(bh_data.keys())}")
if DEBUG:
    for key in bh_data.keys():
        try:
            print(f"{key} type: {bh_data[key][0].dtype}")
        except:
            print(f"{key} type: {type(bh_data[key][0])}")

print(bh_data["sigma_gc"])
print(bh_data["M"])
M_log_mode = jnp.log(bh_data["M"])
M_log_left_edge = jnp.log(bh_data["M"]-bh_data["dM_low"])
M_log_right_edge = jnp.log(bh_data["M"]+bh_data["dM_high"])
sigma_gc_log_mode = jnp.log(bh_data["sigma_gc"])
sigma_gc_log_left_edge = jnp.log(bh_data["sigma_gc"]-bh_data["sigma_gc_low"])
sigma_gc_log_right_edge = jnp.log(bh_data["sigma_gc"]+bh_data["sigma_gc_high"])

Loaded columns: ['Galaxy', 'M', 'dM_low', 'dM_high', 'sigma_gc', 'sigma_gc_low', 'sigma_gc_high']
[134. 186. 202. 128. 175. 312. 320. 204. 217.  69.]
[1.5e+08 8.3e+08 1.5e+08 8.0e+07 1.2e+08 1.5e+09 3.6e+09 5.7e+08 2.1e+09
 4.1e+06]


In [14]:
initial_guess = (M_log_mode[0], 0.5*(M_log_left_edge[0] + M_log_right_edge[0]), 0.0)
solver = optx.LevenbergMarquardt(rtol=1.0e4*MACHINE_EPSILON, atol=1.0e4*MACHINE_EPSILON)
solution = optx.root_find(skewnormal_nonlin_system, solver, initial_guess, args=(M_log_mode[0], M_log_left_edge[0], M_log_right_edge[0]))
print(f"Solution: {solution.value}")
print(f"System at solution: {skewnormal_nonlin_system(solution.value, (M_log_mode[0], M_log_left_edge[0], M_log_right_edge[0]))}")

(M_log_mode[0], M_log_left_edge[0], M_log_right_edge[0])

EquinoxRuntimeError: Above is the stack outside of JIT. Below is the stack inside of JIT:
  File "/home/enrico/miniforge3/envs/jaxns_env/lib/python3.13/site-packages/optimistix/_root_find.py", line 197, in root_find
    sol = least_squares(
        eqx.Partial(_to_lstsq_fn, fn),
    ...<7 lines>...
        throw=throw,
    )
  File "/home/enrico/miniforge3/envs/jaxns_env/lib/python3.13/site-packages/optimistix/_least_squares.py", line 122, in least_squares
    return iterative_solve(
        fn,
    ...<10 lines>...
        rewrite_fn=_rewrite_fn,
    )
  File "/home/enrico/miniforge3/envs/jaxns_env/lib/python3.13/site-packages/optimistix/_iterate.py", line 358, in iterative_solve
    sol = result.error_if(sol, result != RESULTS.successful)
  File "/home/enrico/miniforge3/envs/jaxns_env/lib/python3.13/site-packages/equinox/_module/_prebuilt.py", line 34, in __call__
    return self.__func__(self.__self__, *args, **kwargs)
           ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

equino